In [1]:
# hyperparameter tuning

# Import necessary libraries
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os
import dagshub

c:\Users\krsor\OneDrive\Desktop\mlops\Mini-mlops-project\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow')
dagshub.init(repo_owner='saurav3k2', repo_name='Mini-mlops-project', mlflow=True)


Accessing as saurav3k2

Initialized MLflow to track repo "saurav3k2/Mini-mlops-project"

Repository saurav3k2/Mini-mlops-project initialized!

In [3]:
# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])



In [4]:

def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    return ''.join([char for char in text if not char.isdigit()])

def lower_case(text):
    return " ".join([word.lower() for word in text.split()])

def removing_punctuations(text):
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def normalize_text(df):
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error: {e}')
        raise

In [5]:
# Normalize the text data
df = normalize_text(df)

In [6]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


C:\Users\krsor\AppData\Local\Temp\ipykernel_18980\468518138.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [8]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [9]:
# Set the experiment name
mlflow.set_experiment("LoR-Hyperparameter-Tuning")


2026/04/09 20:06:24 INFO mlflow.tracking.fluent: Experiment with name 'LoR-Hyperparameter-Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/bc143fe226ad4108a71c0d7128c5c1dd', creation_time=1775745387454, experiment_id='3', last_update_time=1775745387454, lifecycle_stage='active', name='LoR-Hyperparameter-Tuning', tags={}, trace_location=None, workspace='default'>

In [10]:
# Define hyperparameter grid for Logistic Regression
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

In [11]:
# Start the parent run for hyperparameter tuning
with mlflow.start_run():

    # Perform grid search
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Log each parameter combination as a child run
    for params, mean_score, std_score in zip(grid_search.cv_results_['params'], grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['std_test_score']):
        with mlflow.start_run(run_name=f"LR with params: {params}", nested=True):
            model = LogisticRegression(**params)
            model.fit(X_train, y_train)
                     
            # Model evaluation
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            
            # Log parameters and metrics
            mlflow.log_params(params)
            mlflow.log_metric("mean_cv_score", mean_score)
            mlflow.log_metric("std_cv_score", std_score)
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            
            
            # Print the results for verification
            print(f"Mean CV Score: {mean_score}, Std CV Score: {std_score}")
            print(f"Accuracy: {accuracy}")
            print(f"Precision: {precision}")
            print(f"Recall: {recall}")
            print(f"F1 Score: {f1}")

    # Log the best run details in the parent run
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    mlflow.log_params(best_params)
    mlflow.log_metric("best_f1_score", best_score)
    
    print(f"Best Params: {best_params}")
    print(f"Best F1 Score: {best_score}")

    # Save and log the notebook
    import os
    notebook_path = "exp3_lor_bow_hp.ipynb"
    os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
    mlflow.log_artifact(notebook_path)

    # Log model
    mlflow.sklearn.log_model(grid_search.best_estimator_, "model")

c:\Users\krsor\OneDrive\Desktop\mlops\Mini-mlops-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\krsor\OneDrive\Desktop\mlops\Mini-mlops-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\krsor\OneDrive\Desktop\mlops\Mini-mlops-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarni

Mean CV Score: 0.7051810430308207, Std CV Score: 0.014168003094990942
Accuracy: 0.7397590361445783
Precision: 0.7752027809965237
Recall: 0.6591133004926109
F1 Score: 0.7124600638977636
🏃 View run LR with params: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3/runs/246a7db7cd5e4db6bd77da60c57eb851
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3


c:\Users\krsor\OneDrive\Desktop\mlops\Mini-mlops-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Mean CV Score: 0.7792109402770179, Std CV Score: 0.010740092596030877
Accuracy: 0.7893975903614457
Precision: 0.7805825242718447
Recall: 0.7921182266009852
F1 Score: 0.7863080684596577
🏃 View run LR with params: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3/runs/41d9c42d865c4d0c9645a15face3829e
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3


c:\Users\krsor\OneDrive\Desktop\mlops\Mini-mlops-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\krsor\OneDrive\Desktop\mlops\Mini-mlops-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


...

Mean CV Score: 0.7712190871481355, Std CV Score: 0.011269036853595352
Accuracy: 0.7826506024096386
Precision: 0.781437125748503
Recall: 0.7714285714285715
F1 Score: 0.7764005949429846
🏃 View run LR with params: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3/runs/d41041f63f1d4c7f965840dd0e2128a8
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3


c:\Users\krsor\OneDrive\Desktop\mlops\Mini-mlops-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Mean CV Score: 0.779151984138562, Std CV Score: 0.00473765823574164
Accuracy: 0.7807228915662651
Precision: 0.7702702702702703
Recall: 0.7862068965517242
F1 Score: 0.7781569965870307
🏃 View run LR with params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3/runs/2637781891b74b49b16a95e6d69355d0
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3
Best Params: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}
Best F1 Score: 0.7904116981776645


2026/04/09 20:08:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 20:08:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run painted-snake-680 at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3/runs/3a19664eac3b419a843eb4d28128225b
🧪 View experiment at: https://dagshub.com/saurav3k2/Mini-mlops-project.mlflow/#/experiments/3
